# Kaggle launcher: one run per session

This notebook holds no project logic. It clones the repository, installs it and calls the scripts.

**Setup:** Accelerator = **GPU T4 x2**, Internet = **on** (Session options, right-hand panel).

**How to use:**
1. Set the values in the first code cell.
2. *Save Version* -> *Save & Run All* (runs in the background, up to Kaggle's session limit).
3. When it finishes, download `<run_id>.zip` from the Output tab and commit its files to `results/runs/<run_id>/`. Update the run's row in `results/run_log.csv` from `run_info.json`.

| What | MODE | ARM / RUN_ID |
| --- | --- | --- |
| Smoke test | `train+eval` | each arm in turn, `CONFIG = "configs/smoke.yaml"` |
| Teacher answers (T2) | `cache` | `CONFIG` only; download the answers file, commit it to `data/` |
| Pilot (T3) | `train+eval` | `RUN_ID = "pilot_<arm>_s3"`, `MAX_STEPS = 100` |
| Main run | `train+eval` | `ARM`, `SEED`; `RUN_ID = None` gives `<arm>_s<seed>` |
| References (E2) | `eval` | `RUN_ID = "ref_teacher"` or `"ref_student0"` |

In the smoke test, the `teacher` arm generates its tiny answers file first if it is missing. Real runs never do: they need the committed `data/teacher_sequences.jsonl`.

In [ ]:
# ---- Settings: the only cell you edit ----
REPO = "https://github.com/redkitex/Knowledge-Distillation-and-Model-Compression.git"
REF = "main"                    # branch or commit; real runs use the commit of the config lock
CONFIG = "configs/smoke.yaml"   # configs/default.yaml for real runs
MODE = "train+eval"             # "train+eval", "eval" (references only) or "cache" (teacher answers)
ARM = "human"                   # human | teacher | onpolicy | ref_hardlabel
SEED = 1
RUN_ID = None                   # None -> f"{ARM}_s{SEED}"; e.g. "pilot_human_s3", "ref_teacher"
ACCOUNT = "T3"                  # whose Kaggle account this is, for the run log
MAX_STEPS = None                # None = full pass; 100 for the pilot

In [ ]:
# Private repository? Add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets) and uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")

# Optional, for faster Hugging Face downloads: a secret named HF_TOKEN.
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
import torch

run_id = RUN_ID or f"{ARM}_s{SEED}"
n_gpus = torch.cuda.device_count()
print("run:", run_id, "| mode:", MODE, "| GPUs:", n_gpus)
assert MODE in ("train+eval", "eval", "cache"), f"unknown MODE {MODE!r}"
if MODE in ("train+eval", "cache"):
    assert n_gpus >= 2, "Needs Accelerator = GPU T4 x2 (teacher on GPU 1, student on GPU 0)"
else:
    assert n_gpus >= 1, "Evaluation needs a GPU"

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q {REPO} repo
%cd /kaggle/working/repo
!git checkout -q {REF} && git log -1 --oneline
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
import os
import yaml

cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
answers = cfg["data"]["teacher_sequences"]

needs_answers = MODE == "cache" or (MODE == "train+eval" and ARM == "teacher")
if needs_answers and (MODE == "cache" or not os.path.exists(answers)):
    if MODE != "cache" and "smoke" not in CONFIG:
        raise FileNotFoundError(f"{answers} is missing. Real runs use the committed file: "
                                "T2 generates it once with MODE = 'cache' and commits it.")
    print("Generating teacher answers ->", answers)
    !python scripts/10_cache_teacher_sequences.py --config {CONFIG}

if MODE == "train+eval":
    steps_flag = f"--max-steps {MAX_STEPS}" if MAX_STEPS else ""
    !python scripts/20_train.py --config {CONFIG} --arm {ARM} --seed {SEED} --run-id {run_id} --account {ACCOUNT} {steps_flag}
if MODE in ("train+eval", "eval"):
    !python scripts/30_evaluate.py --config {CONFIG} --run {run_id}

In [ ]:
# Package the small outputs for download. The checkpoint stays in this session's
# output (results/runs/<run_id>/checkpoint) and never goes into git.
import shutil

if MODE == "cache":
    dst = "/kaggle/working/" + os.path.basename(answers)
    shutil.copy(answers, dst)
    print("Download from the Output tab:", dst, "-> commit it as", answers)
else:
    src = f"results/runs/{run_id}"
    dst = f"/kaggle/working/{run_id}"
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("checkpoint"), dirs_exist_ok=True)
    shutil.make_archive(dst, "zip", dst)
    print("Download from the Output tab:", dst + ".zip")
    !ls -la {dst}